# Getting Census Zipcode Income Breakdowns Data

Getting data from [https://data.census.gov/table?q=ZCTA5+11215&t=Income+and+Poverty&g=010XX00US]

for example looking for 2023 subject table for 11215 zip code

In [ ]:
import os
from dotenv import load_dotenv
import requests
load_dotenv()
census_api_key = os.environ.get("CENSUS_API_KEY")
import pandas as pd

In [ ]:

# random nyc zip code
zcta='11215'
# income breakdown from 2023 group
group = "S0101"

url = f"https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,group(S1901)&for=zip%20code%20tabulation%20area:{zcta}&key={census_api_key}"


response = requests.get(url)

In [ ]:
# Checking if the request was successful
data = None
if response.status_code == 200:
    data = response.json()
else:
    # Handling errors
    print(f"Error: {response.status_code}, {response.text}")

if data:
    data = pd.DataFrame(data)

data.columns = data.iloc[0]
data = data[1:]
# has lots of metrics we don't need
#also isnt human readable yet
data


,NAME,GEO_ID,NAME,S1901_C01_001E,S1901_C01_001EA,S1901_C01_001M,S1901_C01_001MA,S1901_C01_002E,S1901_C01_002EA,S1901_C01_002M,...,S1901_C04_014MA,S1901_C04_015E,S1901_C04_015EA,S1901_C04_015M,S1901_C04_015MA,S1901_C04_016E,S1901_C04_016EA,S1901_C04_016M,S1901_C04_016MA,zip code tabulation area
1,ZCTA5 11215,860Z200US11215,ZCTA5 11215,29747,None,1097,None,3.3,None,0.9,...,(X),-888888888,(X),-888888888,(X),28.4,None,-888888888.0,(X),11215


In [16]:


# Get variable definitions
vars_url = "https://api.census.gov/data/2023/acs/acs5/subject/variables.json"
vars_response = requests.get(vars_url).json()
variables = vars_response['variables']

# match labels for S1901 variables
s1901_labels = {
    var: info['label'] 
    for var, info in variables.items() 
    if var.startswith("S1901_C01_") and var.endswith("E")
}

s1901_labels = s1901_labels.items()
s1901_labels = pd.DataFrame(s1901_labels)
s1901_labels.rename(columns={0: 'code_label', 1: 'readable_label'}, inplace=True)
s1901_labels.columns



Index(['code_label', 'readable_label'], dtype='object')

In [ ]:
col_names = list(s1901_labels['code_label'])
columns_to_select = col_names + ['GEO_ID', 'zip code tabulation area']
filtered_data = data[columns_to_select]
filtered_data = filtered_data.rename(columns={"zip code tabulation area": "ZIP"})
filtered_data

,S1901_C01_016E,S1901_C01_015E,S1901_C01_014E,S1901_C01_013E,S1901_C01_012E,S1901_C01_011E,S1901_C01_010E,S1901_C01_009E,S1901_C01_008E,S1901_C01_007E,S1901_C01_006E,S1901_C01_005E,S1901_C01_004E,S1901_C01_003E,S1901_C01_002E,S1901_C01_001E,GEO_ID,ZIP
1,-888888888,-888888888,29.9,246090,180773,44.5,13.4,15.0,7.0,7.9,2.9,2.3,2.7,1.0,3.3,29747,860Z200US11215,11215


In [21]:
import pandas as pd

def rename_columns_with_labels(data_df, labels_df):
    """
    Rename columns in data_df using human-readable labels from labels_df
    
    Args:
        data_df: DataFrame with code labels as column names
        labels_df: DataFrame with code_label and readable_label columns
    
    Returns:
        DataFrame with renamed columns
    """
    # Check if the expected columns exist in labels_df
    if 'code_label' not in labels_df.columns or 'readable_label' not in labels_df.columns:
        # Print the actual column names for debugging
        print(f"Available columns in labels dataframe: {labels_df.columns.tolist()}")
        raise ValueError("Labels dataframe must contain 'code_label' and 'readable_label' columns")
    
    # Clean the readable labels by replacing '!!' with a single space
    cleaned_labels = labels_df['readable_label'].str.replace('!!', ' ')
    
    # Create a dictionary mapping code labels to cleaned readable labels
    label_dict = dict(zip(labels_df['code_label'], cleaned_labels))
    
    # Create a copy of the original dataframe
    renamed_df = data_df.copy()
    
    # Rename columns that have corresponding readable labels
    columns_to_rename = {col: label_dict.get(col, col) for col in data_df.columns if col in label_dict}
    renamed_df = renamed_df.rename(columns=columns_to_rename)
    
    return renamed_df

# Example usage with your dataframes
# NOTE: Your first dataframe (filtered_data) contains the data, and the second (s1901_labels) contains the labels
# This is the reverse of what was initially assumed
renamed_df = rename_columns_with_labels(filtered_data, s1901_labels)

# Display the result
renamed_df

,Estimate Households PERCENT ALLOCATED Nonfamily income in the past 12 months,Estimate Households PERCENT ALLOCATED Family income in the past 12 months,Estimate Households PERCENT ALLOCATED Household income in the past 12 months,Estimate Households Mean income (dollars),Estimate Households Median income (dollars),"Estimate Households Total $200,000 or more","Estimate Households Total $150,000 to $199,999","Estimate Households Total $100,000 to $149,999","Estimate Households Total $75,000 to $99,999","Estimate Households Total $50,000 to $74,999","Estimate Households Total $35,000 to $49,999","Estimate Households Total $25,000 to $34,999","Estimate Households Total $15,000 to $24,999","Estimate Households Total $10,000 to $14,999","Estimate Households Total Less than $10,000",Estimate Households Total,GEO_ID,ZIP
1,-888888888,-888888888,29.9,246090,180773,44.5,13.4,15.0,7.0,7.9,2.9,2.3,2.7,1.0,3.3,29747,860Z200US11215,11215
